# Fine-tune Qwen3.5-4B on Multimodal-Mind2Web

Trains a web navigation agent on screenshot + task → next action prediction.  
Target hardware: A100 SXM 80GB (RunPod) — or RTX 4060 8GB with `load_in_4bit=True`.  
Output: LoRA adapter + GGUF export for llama.cpp CPU inference.

**Model:** `Qwen/Qwen3.5-4B` — natively multimodal (early fusion), 2.74 GB Q4_K_M GGUF.

In [1]:
import os

# Point HF cache at network volume so downloads survive pod restarts
os.environ["HF_HOME"]            = "/workspace/hf_cache"
os.environ["HF_DATASETS_CACHE"]  = "/workspace/hf_cache/datasets"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"

## 1. Install dependencies

In [ ]:
# Unsloth nightly for latest vision model fixes + deps
!pip install --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade "transformers>=5.1" datasets trl bitsandbytes accelerate sentencepiece pillow
!pip install unsloth_zoo

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"bf16 support:   {torch.cuda.is_bf16_supported()}")

## 2. Load and explore the dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("osunlp/Multimodal-Mind2Web", split="train")
print(f"Training samples: {len(dataset)}")
print(f"Features:         {list(dataset.features.keys())}")

In [ ]:
sample = dataset[0]

print("=== Task ===")
print(sample["confirmed_task"])
print("\n=== Operation ===")
print(sample["operation"])
print("\n=== Value ===")
print(sample.get("value", ""))
print("\n=== Target action ===")
print(sample["target_action_reprs"])
print("\n=== Action history (last 3) ===")
for h in (sample.get("action_reprs") or [])[-3:]:
    print(" -", h)
print(f"\n=== Candidates ===")
print(f"Positive: {len(sample['pos_candidates'])}")
print(f"Negative: {len(sample['neg_candidates'])}")
print("\n=== Screenshot ===")
img = sample["screenshot"]
print(f"Size: {img.size}")
display(img)

## 3. Data conversion

Each sample → Unsloth vision conversation format with **Hermes tool calling**.

**User turn:** screenshot + task + action history + shuffled numbered candidates  
**Assistant turn:** `<tool_call>{"name": "click_element", "arguments": {"index": N}}</tool_call>`

Tools available to the agent:
- `click_element(index)` — CLICK
- `input_text(index, text)` — TYPE
- `select_dropdown_option(index, option)` — SELECT

In [ ]:
import json as _json
import random
from PIL import Image

_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "click_element",
            "description": "Click on a candidate DOM element",
            "parameters": {
                "type": "object",
                "properties": {
                    "index": {"type": "integer", "description": "0-based index of the candidate element"}
                },
                "required": ["index"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "input_text",
            "description": "Type text into an input element",
            "parameters": {
                "type": "object",
                "properties": {
                    "index": {"type": "integer", "description": "0-based index of the input element"},
                    "text":  {"type": "string",  "description": "Text to type"}
                },
                "required": ["index", "text"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "select_dropdown_option",
            "description": "Select an option from a dropdown element",
            "parameters": {
                "type": "object",
                "properties": {
                    "index":  {"type": "integer", "description": "0-based index of the select element"},
                    "option": {"type": "string",  "description": "Option text to select"}
                },
                "required": ["index", "option"]
            }
        }
    }
]

SYSTEM_PROMPT = (
    "You are a web navigation agent. Given a screenshot of a webpage, a task description, "
    "prior action history, and a list of candidate DOM elements, predict the next action "
    "by calling the appropriate tool.\n\n"
    "<tools>\n"
    + _json.dumps(_TOOLS, separators=(",", ":")) +
    "\n</tools>\n\n"
    'Respond with exactly one <tool_call> tag containing a JSON object with "name" and "arguments". '
    "No prose, no explanation."
)

MAX_CANDIDATES = 10
MAX_IMG_SIZE   = 512


def resize_image(img: Image.Image, max_size: int = MAX_IMG_SIZE) -> Image.Image:
    w, h = img.size
    if w <= max_size and h <= max_size:
        return img
    scale = max_size / max(w, h)
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)


def candidate_text(c) -> str:
    if isinstance(c, dict):
        tag   = c.get("tag", "")
        text  = c.get("text", "").strip()
        attrs = c.get("attributes", {})
        aria  = attrs.get("aria-label", "") if isinstance(attrs, dict) else ""
        parts = [p for p in [tag, text or aria] if p]
        return " | ".join(parts) if parts else str(c)
    return str(c)


def operation_to_tool_call(operation: str, index: int, value: str) -> str:
    """Convert Mind2Web operation + candidate index + value → Hermes <tool_call> string."""
    op = operation.upper()
    if op == "TYPE":
        name = "input_text"
        args = {"index": index, "text": value}
    elif op in ("SELECT", "SELECT_OPTION"):
        name = "select_dropdown_option"
        args = {"index": index, "option": value}
    else:  # CLICK and anything else
        name = "click_element"
        args = {"index": index}
    payload = _json.dumps({"name": name, "arguments": args}, separators=(",", ":"))
    return f"<tool_call>{payload}</tool_call>"


def convert_sample(sample: dict) -> dict | None:
    img = sample.get("screenshot")
    if img is None:
        return None

    pos_candidates = sample.get("pos_candidates") or []
    neg_candidates = sample.get("neg_candidates") or []

    if not pos_candidates:
        return None

    pos_candidate = pos_candidates[0]
    pool = [pos_candidate] + neg_candidates[: MAX_CANDIDATES - 1]
    random.shuffle(pool)

    correct_idx = pool.index(pos_candidate)  # 0-based, maps directly to tool arg

    # Display as [0], [1], [2] ... so the index in the label IS the tool argument
    candidate_lines = [f"[{i}] {candidate_text(c)}" for i, c in enumerate(pool)]

    history      = sample.get("action_reprs") or []
    history_text = (
        "\n".join(f"  {i+1}. {h}" for i, h in enumerate(history[-5:]))
        if history else "  (none)"
    )

    user_text = (
        f"TASK: {sample['confirmed_task']}\n\n"
        f"PRIOR ACTIONS:\n{history_text}\n\n"
        "CANDIDATE ELEMENTS:\n" + "\n".join(candidate_lines)
    )

    operation = (sample.get("operation") or "CLICK").upper()
    value     = (sample.get("value") or "").strip()

    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": [
                {"type": "image", "image": resize_image(img.convert("RGB"))},
                {"type": "text",  "text":  user_text},
            ]},
            {"role": "assistant", "content": operation_to_tool_call(operation, correct_idx, value)},
        ]
    }


# Smoke-test
converted = convert_sample(dataset[0])
print("System (first 300 chars):")
print(converted["messages"][0]["content"][:300])
print("\nUser text:")
print(converted["messages"][1]["content"][1]["text"])
print("\nAssistant:")
print(converted["messages"][2]["content"])

## 4. Convert full dataset

In [ ]:
from tqdm.auto import tqdm

random.seed(42)

converted_data = []
skipped        = 0

for sample in tqdm(dataset, desc="Converting samples"):
    result = convert_sample(sample)
    if result is None:
        skipped += 1
    else:
        converted_data.append(result)

print(f"Converted: {len(converted_data):,}  |  Skipped: {skipped}")

## 4b. Recovery Training Data

Generates ~2,000 recovery examples where the agent made a wrong choice and must self-correct.

**Two sources:**
1. **Mind2Web neg-candidate injection** — pretend a distractor was clicked, generate `<think>` + corrective action via Groq (free tier, parallel async)
2. **Tasker DB real failures** — run `mine_tasker_db.py` separately to extract failed→success step pairs from your local Tasker database, then load the output JSONL here

Recovery data is the highest-leverage training signal for small models — even 1–2K examples significantly improves error recognition and self-correction.

In [ ]:
# ── Mine Mind2Web for recovery scenarios ──────────────────────────────────────
# For each sample, pretend a neg_candidate was clicked instead of the correct one.
# The correct action becomes the recovery target.
# No external API needed for this step — purely local data manipulation.

MAX_RECOVERY_SAMPLES = 2000


def create_recovery_scenarios(ds, max_samples: int = MAX_RECOVERY_SAMPLES, seed: int = 42) -> list:
    random.seed(seed)
    scenarios = []

    for sample in tqdm(ds, desc="Mining recovery scenarios"):
        pos = sample.get("pos_candidates") or []
        neg = sample.get("neg_candidates") or []

        if not pos or not neg or sample.get("screenshot") is None:
            continue

        scenarios.append({
            "task":            sample["confirmed_task"],
            "history":         (sample.get("action_reprs") or [])[-3:],
            "wrong_element":   candidate_text(random.choice(neg[:5])),
            "correct_element": candidate_text(pos[0]),
            "operation":       (sample.get("operation") or "CLICK").upper(),
            "value":           (sample.get("value") or "").strip(),
            "screenshot":      sample["screenshot"],
            "pos_candidates":  pos,
            "neg_candidates":  neg,
        })

        if len(scenarios) >= max_samples:
            break

    random.shuffle(scenarios)
    print(f"Recovery scenarios mined: {len(scenarios):,}")
    return scenarios


recovery_scenarios = create_recovery_scenarios(dataset)

s = recovery_scenarios[0]
print(f"\nTask:    {s['task']}")
print(f"Wrong:   {s['wrong_element']}")
print(f"Correct: {s['correct_element']} → {s['operation']}")

In [ ]:
!pip install groq

In [ ]:
# ── Generate <think> blocks via Groq (parallel async) ─────────────────────────
import asyncio
from groq import AsyncGroq
from tqdm.asyncio import tqdm as atqdm

# Free API key: https://console.groq.com
# Set here or: export GROQ_API_KEY=... before launching Jupyter
GROQ_API_KEY     = os.environ.get("GROQ_API_KEY", "your_groq_api_key_here")
# Check https://console.groq.com/docs/models — swap for a Qwen3 model if available
GROQ_MODEL       = "openai/gpt-oss-120b"
GROQ_CONCURRENCY = 5
GROQ_MAX_RETRIES = 5

THINK_PROMPT = """\
You are generating training data for a web navigation AI agent that made a mistake.

Task: {task}
Prior actions: {history}
MISTAKE: The agent just interacted with: "{wrong_element}"
CORRECT action needed: {operation} on "{correct_element}"{value_hint}

Write 2-3 sentences of internal reasoning the agent should have:
1. Recognise what the wrong element was and why it was wrong
2. Identify the correct element and why it is right
3. State the corrective action

Output ONLY the reasoning text. No XML tags, no labels, no preamble."""


async def generate_think(client: AsyncGroq, scenario: dict, semaphore: asyncio.Semaphore) -> str:
    history_str = " → ".join(scenario["history"]) if scenario["history"] else "none"
    value_hint  = f" with value '{scenario['value']}'" if scenario["value"] else ""

    prompt = THINK_PROMPT.format(
        task=scenario["task"],
        history=history_str,
        wrong_element=scenario["wrong_element"],
        correct_element=scenario["correct_element"],
        operation=scenario["operation"],
        value_hint=value_hint,
    )

    async with semaphore:
        for attempt in range(GROQ_MAX_RETRIES):
            try:
                resp = await client.chat.completions.create(
                    model=GROQ_MODEL,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=120,
                    temperature=0.8,
                )
                return resp.choices[0].message.content.strip()
            except Exception as e:
                if "429" in str(e) and attempt < GROQ_MAX_RETRIES - 1:
                    wait = 2 ** attempt + random.uniform(0, 1)
                    await asyncio.sleep(wait)
                    continue
                raise


async def generate_all_thinks(scenarios: list, concurrency: int = GROQ_CONCURRENCY) -> list:
    client    = AsyncGroq(api_key=GROQ_API_KEY)
    semaphore = asyncio.Semaphore(concurrency)
    tasks     = [generate_think(client, s, semaphore) for s in scenarios]
    return await atqdm.gather(*tasks, desc="Generating <think> blocks")


think_blocks = await generate_all_thinks(recovery_scenarios)

print(f"\nGenerated {len(think_blocks)} think blocks")
print("\nExample:")
print(think_blocks[0])

In [ ]:
# ── Convert recovery scenarios → Hermes tool call training format ──────────────

def convert_recovery_sample(scenario: dict, think_text: str) -> dict | None:
    """
    Same structure as convert_sample() but:
    - Injects the wrong action at end of action history
    - Prepends <think> block before the <tool_call>
    """
    pos_candidates = scenario["pos_candidates"]
    neg_candidates = scenario["neg_candidates"]

    if not pos_candidates:
        return None

    pos_candidate = pos_candidates[0]
    pool          = [pos_candidate] + neg_candidates[: MAX_CANDIDATES - 1]
    random.shuffle(pool)

    correct_idx = pool.index(pos_candidate)  # 0-based integer

    candidate_lines = [f"[{i}] {candidate_text(c)}" for i, c in enumerate(pool)]

    history = list(scenario["history"])
    history.append(f"WRONG: interacted with '{scenario['wrong_element']}'")
    history_text = "\n".join(f"  {i+1}. {h}" for i, h in enumerate(history))

    user_text = (
        f"TASK: {scenario['task']}\n\n"
        f"PRIOR ACTIONS (last action was INCORRECT):\n{history_text}\n\n"
        "CANDIDATE ELEMENTS:\n" + "\n".join(candidate_lines)
    )

    operation = scenario["operation"]
    value     = scenario["value"]

    tool_call_str  = operation_to_tool_call(operation, correct_idx, value)
    assistant_text = f"<think>\n{think_text}\n</think>\n{tool_call_str}"

    img_resized = resize_image(scenario["screenshot"].convert("RGB"))

    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": [
                {"type": "image", "image": img_resized},
                {"type": "text",  "text":  user_text},
            ]},
            {"role": "assistant", "content": assistant_text},
        ]
    }


recovery_data = [
    r for r in (
        convert_recovery_sample(s, t)
        for s, t in zip(recovery_scenarios, think_blocks)
    )
    if r is not None
]

print(f"Recovery training examples: {len(recovery_data):,}")
print("\nExample assistant turn:")
print(recovery_data[0]["messages"][2]["content"])

In [ ]:
# ── Merge main + recovery datasets ────────────────────────────────────────────
# Mix 20% recovery into training. More than that hurts normal-action accuracy.
import math

RECOVERY_RATIO = 0.20

n_main     = len(converted_data)
n_recovery = min(
    len(recovery_data),
    math.floor(n_main * RECOVERY_RATIO / (1 - RECOVERY_RATIO)),
)

random.seed(42)
sampled_recovery = random.sample(recovery_data, n_recovery)

combined_data = converted_data + sampled_recovery
random.shuffle(combined_data)

print(f"Main examples:     {n_main:,}")
print(f"Recovery examples: {n_recovery:,}  ({n_recovery / len(combined_data) * 100:.1f}% of total)")
print(f"Total training:    {len(combined_data):,}")

# Overwrite so the training cell picks it up unchanged
converted_data = combined_data

## 5. Load model + LoRA setup

In [ ]:
from unsloth import FastVisionModel

# Qwen3.5-4B: natively multimodal (early fusion) — no separate vision encoder.
# All sizes in the Qwen3.5 family are vision-capable from pretraining.
model, tokenizer = FastVisionModel.from_pretrained(
    model_name                 = "unsloth/Qwen3.5-4B",
    max_seq_length             = 32768,
    load_in_4bit               = False,   # full bf16 — A100 80GB has the headroom
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    # Qwen3.5 uses early fusion, so finetune_vision_layers targets the early
    # vision processing layers. Freeze them; focus LoRA on language layers.
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 16,
    lora_alpha   = 16,
    lora_dropout = 0,
    bias         = "none",
    random_state = 42,
    use_rslora   = False,
)
model.print_trainable_parameters()

## 6. Training

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

USE_BF16 = torch.cuda.is_bf16_supported()
print(f"Training with {'bf16' if USE_BF16 else 'fp16'}")

# max_steps=500 for a quick run (~10-15 min on A100 SXM 80GB for 4B).
# For full epochs: remove max_steps and set num_train_epochs=3
# Full dataset at batch 4 * grad_accum 4 = effective batch 16 ≈ 486 steps/epoch.
MAX_STEPS = 500

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = converted_data,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    args = SFTConfig(
        output_dir                  = "/workspace/qwen35-4b-mind2web-lora",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        gradient_checkpointing      = True,
        learning_rate               = 2e-4,
        lr_scheduler_type           = "cosine",
        warmup_steps                = 20,
        max_steps                   = MAX_STEPS,
        # num_train_epochs            = 3,
        optim                       = "adamw_8bit",
        bf16                        = USE_BF16,
        fp16                        = not USE_BF16,
        logging_steps               = 10,
        save_steps                  = 100,
        save_total_limit            = 3,
        dataset_text_field          = "",
        dataset_kwargs              = {"skip_prepare_dataset": True},
        remove_unused_columns       = False,
        seed                        = 42,
        report_to                   = "none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

## 7. Quick inference test

In [ ]:
import re as _re
FastVisionModel.for_inference(model)

# Held-out sample not seen during 500-step training
test_sample    = dataset[7000]
test_converted = convert_sample(test_sample)

print("Ground truth:", test_converted["messages"][2]["content"])
print()

input_text = tokenizer.apply_chat_template(
    test_converted["messages"][:2],
    add_generation_prompt = True,
    tokenize = False,
)

test_image = test_converted["messages"][1]["content"][0]["image"]

inputs = tokenizer(
    images         = test_image,
    text           = input_text,
    return_tensors = "pt",
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens = 80,
        temperature    = 0.0,
        do_sample      = False,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
prediction    = tokenizer.decode(generated_ids, skip_special_tokens=True)
print("Raw prediction:", prediction)


def parse_tool_call(text: str) -> dict | None:
    m = _re.search(r"<tool_call>(.*?)</tool_call>", text, _re.DOTALL)
    if not m:
        return None
    try:
        return _json.loads(m.group(1).strip())
    except _json.JSONDecodeError:
        return None


parsed = parse_tool_call(prediction)
print("Parsed tool call:", parsed)

## 8. Save LoRA adapter

In [ ]:
ADAPTER_DIR = "/workspace/qwen35-4b-mind2web-lora"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved to {ADAPTER_DIR}")

## 9. Merge LoRA + export to GGUF

Merges weights into a single bf16 model, then quantizes to Q4_K_M (~2.74 GB) and Q8_0 (~4.48 GB) for llama.cpp.

In [ ]:
MERGED_DIR = "/workspace/qwen35-4b-mind2web-merged"

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method = "merged_16bit",
)
print(f"Merged model saved to {MERGED_DIR}")

In [ ]:
# Q4_K_M — ~2.74 GB, fastest CPU inference
model.save_pretrained_gguf(
    "/workspace/qwen35-4b-mind2web-q4",
    tokenizer,
    quantization_method = "q4_k_m",
)
print("Q4_K_M GGUF saved to /workspace/qwen35-4b-mind2web-q4/")

In [ ]:
# Q8_0 — ~4.48 GB, higher accuracy
model.save_pretrained_gguf(
    "/workspace/qwen35-4b-mind2web-q8",
    tokenizer,
    quantization_method = "q8_0",
)
print("Q8_0 GGUF saved to /workspace/qwen35-4b-mind2web-q8/")

## 9b. Push to Hugging Face Hub

Uploads the LoRA adapter, merged model, and GGUF quantizations to your HF account.  
Set your token via `huggingface-cli login` or the `HF_TOKEN` env var.

In [ ]:
from huggingface_hub import HfApi, login

# Authenticate — will prompt for token if not already logged in
login()

api = HfApi()

# ── Change this to your HF username / org ──
HF_USERNAME = "your-hf-username"
REPO_PREFIX = f"{HF_USERNAME}/qwen35-4b-mind2web"

# Upload LoRA adapter
api.create_repo(f"{REPO_PREFIX}-lora", exist_ok=True)
api.upload_folder(
    folder_path = ADAPTER_DIR,
    repo_id     = f"{REPO_PREFIX}-lora",
    commit_message = "Upload LoRA adapter",
)
print(f"LoRA adapter uploaded to https://huggingface.co/{REPO_PREFIX}-lora")

# Upload merged bf16 model
api.create_repo(f"{REPO_PREFIX}-merged", exist_ok=True)
api.upload_folder(
    folder_path = MERGED_DIR,
    repo_id     = f"{REPO_PREFIX}-merged",
    commit_message = "Upload merged bf16 model",
)
print(f"Merged model uploaded to https://huggingface.co/{REPO_PREFIX}-merged")

# Upload GGUF quantizations
for quant, path in [("q4", "/workspace/qwen35-4b-mind2web-q4"), ("q8", "/workspace/qwen35-4b-mind2web-q8")]:
    repo_id = f"{REPO_PREFIX}-gguf-{quant}"
    api.create_repo(repo_id, exist_ok=True)
    api.upload_folder(
        folder_path    = path,
        repo_id        = repo_id,
        commit_message = f"Upload {quant.upper()} GGUF",
    )
    print(f"GGUF {quant.upper()} uploaded to https://huggingface.co/{repo_id}")

## 10. CPU inference with llama.cpp

### Build llama.cpp

```bash
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
cmake -B build -DLLAMA_CURL=ON
cmake --build build --config Release -j$(nproc)
```

### Run inference

```bash
ls /workspace/qwen35-4b-mind2web-q4/   # find the .gguf and mmproj files

./build/bin/llama-llava-cli \
  --model   /workspace/qwen35-4b-mind2web-q4/model-q4_k_m.gguf \
  --mmproj  /workspace/qwen35-4b-mind2web-q4/mmproj-model-f16.gguf \
  --image   /path/to/screenshot.png \
  --ctx-size 32768 \
  --n-predict 80 \
  --threads   8
```

### Python wrapper (llama-cpp-python)

```bash
pip install llama-cpp-python
```

```python
import json, re, base64
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Llava15ChatHandler

_TOOLS = [
    {"type": "function", "function": {"name": "click_element", "description": "Click on a candidate DOM element", "parameters": {"type": "object", "properties": {"index": {"type": "integer"}}, "required": ["index"]}}},
    {"type": "function", "function": {"name": "input_text",    "description": "Type text into an input element",   "parameters": {"type": "object", "properties": {"index": {"type": "integer"}, "text": {"type": "string"}},  "required": ["index", "text"]}}},
    {"type": "function", "function": {"name": "select_dropdown_option", "description": "Select a dropdown option", "parameters": {"type": "object", "properties": {"index": {"type": "integer"}, "option": {"type": "string"}}, "required": ["index", "option"]}}},
]

SYSTEM_PROMPT = (
    "You are a web navigation agent. Given a screenshot of a webpage, a task description, "
    "prior action history, and a list of candidate DOM elements, predict the next action "
    "by calling the appropriate tool.\n\n"
    "<tools>\n" + json.dumps(_TOOLS, separators=(",", ":")) + "\n</tools>\n\n"
    'Respond with exactly one <tool_call> tag containing a JSON object with "name" and "arguments". '
    "No prose, no explanation."
)

chat_handler = Llava15ChatHandler(clip_model_path="qwen35-4b-mind2web-q4/mmproj-model-f16.gguf")
llm = Llama(
    model_path   = "qwen35-4b-mind2web-q4/model-q4_k_m.gguf",
    chat_handler = chat_handler,
    n_ctx        = 32768,
    n_threads    = 8,
    verbose      = False,
)

def image_to_data_uri(path: str) -> str:
    with open(path, "rb") as f:
        return f"data:image/png;base64,{base64.b64encode(f.read()).decode()}"

# Candidates displayed as [0], [1], [2] ... index maps directly to tool arguments
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": image_to_data_uri("screenshot.png")}},
            {"type": "text", "text": "TASK: ...\nPRIOR ACTIONS: ...\nCANDIDATE ELEMENTS:\n[0] button | Sign in\n[1] input | Email"},
        ]},
    ],
    max_tokens  = 80,
    temperature = 0.0,
)
raw = response["choices"][0]["message"]["content"]
print("Raw:", raw)
```

### Parse the tool call output

```python
def parse_tool_call(text: str) -> dict | None:
    m = re.search(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(1).strip())
    except json.JSONDecodeError:
        return None

tool_call = parse_tool_call(raw)
# Example outputs:
# {"name": "click_element",          "arguments": {"index": 0}}
# {"name": "input_text",             "arguments": {"index": 1, "text": "hello@example.com"}}
# {"name": "select_dropdown_option", "arguments": {"index": 2, "option": "New York"}}
print("Tool call:", tool_call)
```

## 11. Tool Format Validation

Runs the fine-tuned model on held-out samples covering CLICK, TYPE, and SELECT operations.  
Checks that every output is a valid, parseable Hermes `<tool_call>` with correct structure.

In [ ]:
import re, json

VALID_TOOLS = {"click_element", "input_text", "select_dropdown_option"}
REQUIRED_ARGS = {
    "click_element":          {"index"},
    "input_text":             {"index", "text"},
    "select_dropdown_option": {"index", "option"},
}

FastVisionModel.for_inference(model)


def run_inference(sample: dict) -> str:
    """Run model on a single converted sample, return raw generated text."""
    prompt = tokenizer.apply_chat_template(
        sample["messages"][:2],
        add_generation_prompt=True,
        tokenize=False,
    )
    image = sample["messages"][1]["content"][0]["image"]
    inputs = tokenizer(images=image, text=prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, temperature=0.0, do_sample=False)
    generated = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def validate_tool_call(raw: str) -> tuple[bool, str, dict | None]:
    """
    Returns (passed, reason, parsed_call).
    Checks:
      1. Contains exactly one <tool_call>...</tool_call> block
      2. Inner content is valid JSON
      3. Has 'name' and 'arguments' keys
      4. 'name' is a known tool
      5. 'arguments' contains all required keys
      6. 'index' is an integer
    """
    matches = re.findall(r"<tool_call>(.*?)</tool_call>", raw, re.DOTALL)

    if len(matches) == 0:
        return False, "No <tool_call> tag found", None
    if len(matches) > 1:
        return False, f"Multiple <tool_call> tags ({len(matches)})", None

    try:
        call = json.loads(matches[0].strip())
    except json.JSONDecodeError as e:
        return False, f"Invalid JSON: {e}", None

    if "name" not in call:
        return False, "Missing 'name' key", call
    if "arguments" not in call:
        return False, "Missing 'arguments' key", call

    name = call["name"]
    args = call["arguments"]

    if name not in VALID_TOOLS:
        return False, f"Unknown tool '{name}'", call

    required = REQUIRED_ARGS[name]
    missing  = required - set(args.keys())
    if missing:
        return False, f"Missing argument(s): {missing}", call

    if not isinstance(args.get("index"), int):
        return False, f"'index' must be int, got {type(args.get('index')).__name__}", call

    return True, "OK", call


# ── Find held-out samples for each operation type ─────────────────────────────
def find_sample_by_op(ds, operation: str, start: int = 7000, limit: int = 200) -> dict | None:
    for i in range(start, min(start + limit, len(ds))):
        s = ds[i]
        if (s.get("operation") or "CLICK").upper() == operation:
            converted = convert_sample(s)
            if converted:
                return converted
    return None

test_cases = []
for op in ("CLICK", "TYPE", "SELECT"):
    sample = find_sample_by_op(dataset, op)
    if sample:
        test_cases.append((op, sample))
    else:
        print(f"  Warning: no {op} sample found in held-out range")

# ── Run tests ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("TOOL FORMAT VALIDATION")
print("=" * 60)

passed = 0
for op, sample in test_cases:
    ground_truth = sample["messages"][2]["content"]
    raw          = run_inference(sample)
    ok, reason, parsed = validate_tool_call(raw)

    status = "✓ PASS" if ok else "✗ FAIL"
    if ok:
        passed += 1

    print(f"\n[{status}] Operation: {op}")
    print(f"  Ground truth : {ground_truth}")
    print(f"  Raw output   : {raw}")
    print(f"  Parsed call  : {parsed}")
    if not ok:
        print(f"  Reason       : {reason}")

print("\n" + "=" * 60)
print(f"Result: {passed}/{len(test_cases)} passed")
if passed == len(test_cases):
    print("All tool calls are correctly formatted for vLLM hermes parser ✓")
else:
    print("Some tool calls failed format validation — check output above")